In [18]:
import os
import numpy as np
import pandas as pd


def generate_realized_cov_proxy(
    H,
    in_path="../data/log_returns.csv",
    out_dir="../proxy",
    prefix="realized_cov",
    debug=True
):

    # -----------------------
    # Load returns
    # -----------------------
    df = pd.read_csv(in_path, parse_dates=["Date"]).set_index("Date").sort_index()
    assets = df.columns.tolist()

    R = df.to_numpy()
    dates = df.index
    T, N = R.shape

    # -----------------------
    # Full matrix column names
    # -----------------------
    colnames = [
        f"cov_{assets[i]}__{assets[j]}"
        for i in range(N)
        for j in range(N)
    ]

    rows = []
    out_dates = []

    # -----------------------
    # Compute proxy
    # window = t ... t+H-1
    # -----------------------
    for k in range(T - H + 1):

        window = R[k:k+H]
        Sigma = (window.T @ window) / H

        # enforce symmetry
        Sigma = 0.5 * (Sigma + Sigma.T)

        rows.append(Sigma.flatten())
        out_dates.append(dates[k])

    proxy_df = pd.DataFrame(rows, index=out_dates, columns=colnames)
    proxy_df.index.name = "Date"

    # -----------------------
    # Save
    # -----------------------
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{prefix}_h{H}.csv")
    proxy_df.to_csv(out_path)

    print("Saved:", out_path)
    print("Proxy shape:", proxy_df.shape)
    print("Proxy date range:", proxy_df.index.min().date(), "->", proxy_df.index.max().date())

    # -----------------------
    # Optional sanity check
    # -----------------------
    if debug:
        k = 0
        print("\nSanity check:")
        print("t:", dates[k].date())
        print("window start:", dates[k].date())
        print("window end:", dates[k+H-1].date())

        manual = (R[k:k+H].T @ R[k:k+H]) / H
        stored = proxy_df.iloc[k].to_numpy().reshape(N, N)

        diff = np.abs(manual - stored).max()
        print("max difference manual vs stored:", diff)

    return proxy_df

In [19]:
proxy_21 = generate_realized_cov_proxy(H=21)
proxy_21.head()

Saved: ../proxy/realized_cov_h21.csv
Proxy shape: (2184, 64)
Proxy date range: 2017-03-28 -> 2025-12-02

Sanity check:
t: 2017-03-28
window start: 2017-03-28
window end: 2017-04-26
max difference manual vs stored: 0.0


,cov_US_EQUITY__US_EQUITY,cov_US_EQUITY__INTL_EQUITY,cov_US_EQUITY__US_BONDS,cov_US_EQUITY__INTL_BONDS,cov_US_EQUITY__US_REIT,cov_US_EQUITY__INTL_REIT,cov_US_EQUITY__GOLD,cov_US_EQUITY__BTC,cov_INTL_EQUITY__US_EQUITY,cov_INTL_EQUITY__INTL_EQUITY,...,cov_GOLD__GOLD,cov_GOLD__BTC,cov_BTC__US_EQUITY,cov_BTC__INTL_EQUITY,cov_BTC__US_BONDS,cov_BTC__INTL_BONDS,cov_BTC__US_REIT,cov_BTC__INTL_REIT,cov_BTC__GOLD,cov_BTC__BTC
Date,,,,,,,,,,,,,,,,,,,,,
2017-03-28,0.000021,0.000016,-0.000005,-0.000002,0.000006,0.000008,-0.000015,0.000032,0.000016,0.000033,...,0.000034,0.000007,0.000032,0.000003,0.000013,0.000004,0.000040,0.000044,0.000007,0.000559
2017-03-29,0.000019,0.000016,-0.000004,-0.000002,0.000004,0.000008,-0.000014,0.000033,0.000016,0.000033,...,0.000034,0.000002,0.000033,0.000003,0.000014,0.000009,0.000041,0.000046,0.000002,0.000597
2017-03-30,0.000019,0.000015,-0.000005,-0.000002,0.000005,0.000009,-0.000015,0.000033,0.000015,0.000033,...,0.000034,0.000003,0.000033,0.000005,0.000015,0.000010,0.000043,0.000047,0.000003,0.000595
2017-03-31,0.000019,0.000016,-0.000005,-0.000002,0.000005,0.000010,-0.000015,0.000045,0.000016,0.000034,...,0.000035,-0.000038,0.000045,0.000018,0.000004,0.000005,0.000067,0.000052,-0.000038,0.000868
2017-04-03,0.000019,0.000016,-0.000005,-0.000002,0.000006,0.000010,-0.000014,0.000049,0.000016,0.000034,...,0.000035,-0.000044,0.000049,0.000027,0.000004,0.000008,0.000054,0.000056,-0.000044,0.000801
